In [7]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast, GradScaler
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
import timm

from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/TimeSeriesDeepLearning_FIM601/kaggle_data/optiver-realized-volatility-prediction'
DIR = '/content/drive/MyDrive/TimeSeriesDeepLearning_FIM601/'
LOCAL_DATA_DIR = '/content/data'

!cp -r {DATA_DIR} {LOCAL_DATA_DIR}


class OrderFlowDataset(Dataset):
    def __init__(self, target_csv_path: str, book_path: str, trade_path: str, transform=None):
        self.target = pd.read_csv(target_csv_path)
        self.index_map = self.target[['stock_id', 'time_id']].to_numpy()
        self.target = self.target.to_numpy()

        self.transform = transform

        full_book = pd.read_parquet(book_path)
        full_trade = pd.read_parquet(trade_path)

        self.books = {k: v for k, v in full_book.groupby(['stock_id', 'time_id'])}
        self.trades = {k: v for k, v in full_trade.groupby(['stock_id', 'time_id'])}

    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        stock_id, time_id = self.index_map[idx]
        book_data = self.books.get((stock_id, time_id), pd.DataFrame())
        trade_data = self.trades.get((stock_id, time_id), pd.DataFrame())
        target = self.target[idx, 2]

        sample = {'book': book_data, 'trade': trade_data, 'r_vol': target}
        if self.transform:
            sample = self.transform(sample)
        return sample


class ToImage(object):
    def __init__(self, output_size=(600, 600, 4), bounds='auto', pad=True, include_features=True):
        self.output_size = output_size
        self.bounds = bounds
        self.pad = pad
        self.include_features = include_features

    @staticmethod
    def _safe_std(values: np.ndarray) -> float:
        if values.size < 2:
            return 0.0
        return float(np.std(values, ddof=1))

    def _extract_engineered_features(self, book: pd.DataFrame, trade: pd.DataFrame) -> np.ndarray:
        if book.empty:
            return np.zeros(16, dtype=np.float32)

        bid_price1 = book['bid_price1'].to_numpy(dtype=np.float64)
        ask_price1 = book['ask_price1'].to_numpy(dtype=np.float64)
        bid_size1 = book['bid_size1'].to_numpy(dtype=np.float64)
        ask_size1 = book['ask_size1'].to_numpy(dtype=np.float64)
        bid_size2 = book['bid_size2'].to_numpy(dtype=np.float64)
        ask_size2 = book['ask_size2'].to_numpy(dtype=np.float64)
        seconds = book['seconds_in_bucket'].to_numpy(dtype=np.int32)

        denom = bid_size1 + ask_size1 + 1e-8
        wap1 = (bid_price1 * ask_size1 + ask_price1 * bid_size1) / denom
        spread1 = ask_price1 - bid_price1

        wap1_clipped = np.clip(wap1, 1e-8, None)
        if wap1_clipped.size > 1:
            log_ret = np.diff(np.log(wap1_clipped))
            realized_vol = float(np.sqrt(np.sum(log_ret * log_ret)))
        else:
            realized_vol = 0.0

        vol_l1 = bid_size1 + ask_size1
        vol_l2 = bid_size2 + ask_size2
        vol_total = vol_l1 + vol_l2
        vol_imbalance = (bid_size1 - ask_size1) / (bid_size1 + ask_size1 + 1e-8)

        book_update_count = float(len(book))
        unique_seconds_count = float(np.unique(seconds).size)
        mean_spread = float(np.mean(spread1))
        std_spread = self._safe_std(spread1)
        wap1_mean = float(np.mean(wap1))
        vol_sum = float(np.sum(vol_total))
        vol_q90 = float(np.quantile(vol_total, 0.9)) if vol_total.size else 0.0
        vol_imbalance_mean = float(np.mean(vol_imbalance))
        vol_imbalance_std = self._safe_std(vol_imbalance)

        trade_count = 0.0
        trade_size_sum = 0.0
        trade_order_count_sum = 0.0
        trade_avg_size_per_order = 0.0
        trade_inter_arrival_mean = 0.0
        trade_price_return_std = 0.0

        if not trade.empty:
            trade_count = float(len(trade))
            trade_seconds = trade['seconds_in_bucket'].to_numpy(dtype=np.int32)
            trade_size = trade['size'].to_numpy(dtype=np.float64)
            trade_order_count = trade['order_count'].to_numpy(dtype=np.float64)
            trade_price = trade['price'].to_numpy(dtype=np.float64)

            trade_size_sum = float(np.sum(trade_size))
            trade_order_count_sum = float(np.sum(trade_order_count))
            trade_avg_size_per_order = float(np.mean(trade_size / (trade_order_count + 1e-8)))

            if trade_seconds.size > 1:
                trade_inter_arrival_mean = float(np.mean(np.diff(trade_seconds)))

            if trade_price.size > 1:
                trade_price_ret = np.diff(np.log(np.clip(trade_price, 1e-8, None)))
                trade_price_return_std = self._safe_std(trade_price_ret)

        features = np.array([
            book_update_count, unique_seconds_count, mean_spread, std_spread, wap1_mean, realized_vol,
            vol_sum, vol_q90, vol_imbalance_mean, vol_imbalance_std, trade_count, trade_size_sum,
            trade_order_count_sum, trade_avg_size_per_order, trade_inter_arrival_mean, trade_price_return_std,
        ], dtype=np.float32)
        return features

    def __call__(self, sample):
        book, trade, r_vol = sample['book'], sample['trade'], sample['r_vol']
        n_time, n_price, n_channels = self.output_size
        image = np.zeros((n_time, n_price, n_channels), dtype=np.int32)
        features = self._extract_engineered_features(book, trade) if self.include_features else None
        if book.empty:
            output = {'image': image, 'r_vol': r_vol}
            if features is not None:
                output['features'] = features
            return output

        if self.bounds == 'auto':
            min_price = min(book['bid_price1'].min(), book['bid_price2'].min(), book['ask_price1'].min(), book['ask_price2'].min())
            max_price = max(book['bid_price1'].max(), book['bid_price2'].max(), book['ask_price1'].max(), book['ask_price2'].max())
            if not trade.empty:
                min_price = min(min_price, trade['price'].min())
                max_price = max(max_price, trade['price'].max())
            padding = 0.001 * (max_price - min_price + 1e-8)
            bounds = (float(min_price - padding), float(max_price + padding))
        else:
            bounds = (float(self.bounds[0]), float(self.bounds[1]))

        price_edges = np.linspace(bounds[0], bounds[1], n_price + 1)

        bid_bin_1 = np.searchsorted(price_edges, book['bid_price1'].to_numpy()) - 1
        bid_bin_2 = np.searchsorted(price_edges, book['bid_price2'].to_numpy()) - 1
        ask_bin_1 = np.searchsorted(price_edges, book['ask_price1'].to_numpy()) - 1
        ask_bin_2 = np.searchsorted(price_edges, book['ask_price2'].to_numpy()) - 1

        bid_bin_1 = np.clip(bid_bin_1, 0, n_price - 1)
        bid_bin_2 = np.clip(bid_bin_2, 0, n_price - 1)
        ask_bin_1 = np.clip(ask_bin_1, 0, n_price - 1)
        ask_bin_2 = np.clip(ask_bin_2, 0, n_price - 1)

        sec = book['seconds_in_bucket'].to_numpy(dtype=np.int32)
        bs1 = book['bid_size1'].to_numpy(dtype=np.int32)
        bs2 = book['bid_size2'].to_numpy(dtype=np.int32)
        as1 = book['ask_size1'].to_numpy(dtype=np.int32)
        as2 = book['ask_size2'].to_numpy(dtype=np.int32)

        np.add.at(image, (sec, bid_bin_1, 0), bs1)
        np.add.at(image, (sec, bid_bin_2, 0), bs2)
        np.add.at(image, (sec, ask_bin_1, 1), as1)
        np.add.at(image, (sec, ask_bin_2, 1), as2)

        if not trade.empty:
            trade_sec = trade['seconds_in_bucket'].to_numpy(dtype=np.int32)
            trade_bin = np.searchsorted(price_edges, trade['price'].to_numpy(), side='right') - 1
            trade_bin = np.clip(trade_bin, 0, n_price - 1)
            trade_size = trade['size'].to_numpy(dtype=np.int32)
            trade_oc = trade['order_count'].to_numpy(dtype=np.int32)

            np.add.at(image[:, :, 2], (trade_sec, trade_bin), trade_size)
            if self.pad:
                left_mask = trade_bin - 1 >= 0
                right_mask = trade_bin + 1 < n_price
                np.add.at(image[:, :, 2], (trade_sec[left_mask], trade_bin[left_mask] - 1), trade_size[left_mask])
                np.add.at(image[:, :, 2], (trade_sec[right_mask], trade_bin[right_mask] + 1), trade_size[right_mask])
            np.add.at(image[:, :, 3], (trade_sec, trade_bin), trade_oc)

        output = {'image': image, 'r_vol': r_vol}
        if features is not None:
            output['features'] = features
        return output


class OrderFlowRegressor(nn.Module):
    def __init__(self, tabular_dim):
        super().__init__()
        self.image_encoder = timm.create_model(
            'vit_base_patch16_224',
            pretrained=True,
            in_chans=4,
            num_classes=0,
            global_pool='avg',
            dynamic_img_size=True
        )
        image_dim = self.image_encoder.num_features

        self.feature_encoder = nn.Sequential(
            nn.LayerNorm(tabular_dim),
            nn.Linear(tabular_dim, 128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 128),
            nn.GELU(),
        )

        self.regression_head = nn.Sequential(
            nn.Linear(image_dim + 128, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
        )

    def forward(self, image, features):
        H, W = image.shape[2], image.shape[3]
        pad_h = (16 - H % 16) % 16
        pad_w = (16 - W % 16) % 16
        image = F.pad(image, (0, pad_w, 0, pad_h))

        image_embedding = self.image_encoder(image)
        feature_embedding = self.feature_encoder(features)
        fused = torch.cat([image_embedding, feature_embedding], dim=1)
        return self.regression_head(fused).squeeze(-1)


class NormalizedSubset(Dataset):
    def __init__(self, base_dataset, indices, img_mean, img_std, feat_mean, feat_std):
        self.base_dataset = base_dataset
        self.indices = np.asarray(indices, dtype=np.int64)
        self.img_mean = img_mean.astype(np.float32)
        self.img_std = img_std.astype(np.float32)
        self.feat_mean = feat_mean.astype(np.float32)
        self.feat_std = feat_std.astype(np.float32)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        sample = self.base_dataset[int(self.indices[i])]
        image = np.log1p(sample['image'].astype(np.float32))
        image = (image - self.img_mean[None, None, :]) / self.img_std[None, None, :]
        image = np.transpose(image, (2, 0, 1)).copy()
        features = sample['features'].astype(np.float32)
        features = (features - self.feat_mean) / self.feat_std
        target = np.float32(sample['r_vol'])
        return {
            'image': torch.from_numpy(image),
            'features': torch.from_numpy(features),
            'r_vol': torch.tensor(target, dtype=torch.float32),
        }

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
from google.colab import drive
drive.mount('/content/drive')
!find /content/drive/MyDrive -name "ImageEncoding.py" | head -n 20

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
!find /content -maxdepth 4 -name "ImageEncoding.py" | head -n 20
!pwd
!ls -la

/content
total 20
drwxr-xr-x 1 root root 4096 Apr 14 10:55 .
drwxr-xr-x 1 root root 4096 Apr 14 10:40 ..
drwxr-xr-x 4 root root 4096 Mar 30 13:34 .config
drwx------ 6 root root 4096 Apr 14 10:55 drive
drwxr-xr-x 1 root root 4096 Mar 30 13:34 sample_data


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

img_transform = ToImage(output_size=(600, 80, 4), include_features=True)
base_dataset = OrderFlowDataset(
    f"{LOCAL_DATA_DIR}/train.csv",
    f"{LOCAL_DATA_DIR}/book_train.parquet",
    f"{LOCAL_DATA_DIR}/trade_train.parquet",
    transform=img_transform
)
feature_dim = int(base_dataset[0]["features"].shape[0])
index_df = pd.DataFrame(base_dataset.index_map, columns=["stock_id", "time_id"])

cpu_count = os.cpu_count() or 0
hyperparameters = {
    "batch_size": 512,
    "num_workers": max(0, cpu_count - 1),
    "pin_memory": True,
    "prefetch_factor": 2
}

cv_config = {
    "n_splits": 5,
    "gap_time_ids": 2,
    "num_epochs": 6,
    "stats_max_samples": 8000,
    "early_stopping_patience": 2,
    "early_stopping_min_delta": 0.0
}

/tmp/ipykernel_3447/474554348.py:34: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  self.books = {k: v for k, v in full_book.groupby(['stock_id', 'time_id'])}
/tmp/ipykernel_3447/474554348.py:35: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  self.trades = {k: v for k, v in full_trade.groupby(['stock_id', 'time_id'])}


In [13]:
# Active settings: full training.
hyperparameters['batch_size'] = 512
hyperparameters['num_workers'] = max(0, (os.cpu_count() or 0) - 1)
cv_config['n_splits'] = 5
cv_config['gap_time_ids'] = 2
cv_config['num_epochs'] = 6
cv_config['stats_max_samples'] = 8000

print('Using full training settings:', hyperparameters, cv_config)

Using full training settings: {'batch_size': 512, 'num_workers': 11, 'pin_memory': True, 'prefetch_factor': 2} {'n_splits': 5, 'gap_time_ids': 2, 'num_epochs': 6, 'stats_max_samples': 8000}


In [10]:
def make_purged_expanding_folds(time_ids, n_splits=5, gap_time_ids=2):
    unique_times = np.array(sorted(np.unique(time_ids)))
    n_times = len(unique_times)
    val_block = max(1, n_times // (n_splits + 1))
    folds = []

    for fold in range(n_splits):
        train_end = val_block * (fold + 1)
        val_start = train_end + gap_time_ids
        val_end = min(val_start + val_block, n_times)

        if val_start >= n_times or val_end <= val_start:
            continue

        train_times = unique_times[:train_end]
        val_times = unique_times[val_start:val_end]

        if len(train_times) == 0 or len(val_times) == 0:
            continue

        folds.append((train_times, val_times))
    return folds


def compute_global_stats(dataset, sample_indices, max_samples=8000):
    if len(sample_indices) == 0:
        raise ValueError("sample_indices must not be empty")

    rng = np.random.default_rng(42)
    sample_indices = np.asarray(sample_indices, dtype=np.int64)
    if len(sample_indices) > max_samples:
        sample_indices = rng.choice(sample_indices, size=max_samples, replace=False)

    first = dataset[int(sample_indices[0])]
    channels = int(first['image'].shape[-1])
    feat_dim = int(first['features'].shape[0])

    img_sum = np.zeros(channels, dtype=np.float64)
    img_sq_sum = np.zeros(channels, dtype=np.float64)
    img_count = 0

    feat_sum = np.zeros(feat_dim, dtype=np.float64)
    feat_sq_sum = np.zeros(feat_dim, dtype=np.float64)
    feat_count = 0

    for idx in sample_indices:
        sample = dataset[int(idx)]
        image = np.log1p(sample['image'].astype(np.float32))
        image = image.reshape(-1, channels)
        img_sum += image.sum(axis=0)
        img_sq_sum += np.square(image).sum(axis=0)
        img_count += image.shape[0]

        feat = sample['features'].astype(np.float32)
        feat_sum += feat
        feat_sq_sum += np.square(feat)
        feat_count += 1

    img_mean = img_sum / img_count
    img_var = np.maximum((img_sq_sum / img_count) - np.square(img_mean), 1e-6)
    img_std = np.sqrt(img_var)

    feat_mean = feat_sum / feat_count
    feat_var = np.maximum((feat_sq_sum / feat_count) - np.square(feat_mean), 1e-6)
    feat_std = np.sqrt(feat_var)

    return img_mean.astype(np.float32), img_std.astype(np.float32), feat_mean.astype(np.float32), feat_std.astype(np.float32)


def train_one_fold(fold_id, train_indices, val_indices):
    img_mean, img_std, feat_mean, feat_std = compute_global_stats(
        base_dataset,
        train_indices,
        max_samples=cv_config['stats_max_samples']
    )

    train_ds = NormalizedSubset(base_dataset, train_indices, img_mean, img_std, feat_mean, feat_std)
    val_ds = NormalizedSubset(base_dataset, val_indices, img_mean, img_std, feat_mean, feat_std)

    train_loader = DataLoader(train_ds, shuffle=True, **hyperparameters)
    val_loader = DataLoader(val_ds, shuffle=False, **hyperparameters)

    model = OrderFlowRegressor(tabular_dim=feature_dim).to(device)
    criterion = nn.HuberLoss()

    optimizer_adamw = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

    num_epochs = cv_config['num_epochs']
    total_steps = len(train_loader) * num_epochs
    warmup_steps = max(1, int(0.1 * total_steps))
    decay_steps = max(1, total_steps - warmup_steps)

    warmup = LinearLR(optimizer_adamw, start_factor=0.01, end_factor=1.0, total_iters=warmup_steps)
    cosine = CosineAnnealingLR(optimizer_adamw, T_max=decay_steps, eta_min=1e-6)
    scheduler = SequentialLR(optimizer_adamw, schedulers=[warmup, cosine], milestones=[warmup_steps])
    scaler = GradScaler('cuda')

    best_val = float('inf')
    best_epoch = -1
    bad_epochs = 0
    patience = int(cv_config.get('early_stopping_patience', 2))
    min_delta = float(cv_config.get('early_stopping_min_delta', 0.0))
    best_model_path = f"{DIR}/best_model_fold_{fold_id}.pth"
    fold_history = []

    for epoch in range(num_epochs):
        model.train()
        train_loss_sum = 0.0

        for batch in train_loader:
            images = batch['image'].to(device, dtype=torch.float32, non_blocking=True)
            features = batch['features'].to(device, dtype=torch.float32, non_blocking=True)
            targets = batch['r_vol'].to(device, dtype=torch.float32, non_blocking=True)

            optimizer_adamw.zero_grad()

            with autocast('cuda', dtype=torch.bfloat16):
                preds = model(images, features)
                loss = criterion(preds, targets)

            scaler.scale(loss).backward()
            scaler.step(optimizer_adamw)
            scaler.update()
            scheduler.step()

            train_loss_sum += loss.item() * images.size(0)

        train_loss = train_loss_sum / len(train_ds)

        model.eval()
        val_loss_sum = 0.0
        with torch.no_grad():
            for batch in val_loader:
                images = batch['image'].to(device, dtype=torch.float32, non_blocking=True)
                features = batch['features'].to(device, dtype=torch.float32, non_blocking=True)
                targets = batch['r_vol'].to(device, dtype=torch.float32, non_blocking=True)

                with autocast('cuda', dtype=torch.bfloat16):
                    preds = model(images, features)
                    loss = criterion(preds, targets)

                val_loss_sum += loss.item() * images.size(0)

        val_loss = val_loss_sum / len(val_ds)
        fold_history.append({
            'fold': fold_id,
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'val_loss': val_loss,
        })

        improved = val_loss < (best_val - min_delta)
        if improved:
            best_val = val_loss
            best_epoch = epoch + 1
            bad_epochs = 0
            torch.save({
                'fold': fold_id,
                'epoch': best_epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer_adamw.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'scaler_state_dict': scaler.state_dict(),
                'best_val_loss': best_val,
                'img_mean': img_mean,
                'img_std': img_std,
                'feat_mean': feat_mean,
                'feat_std': feat_std,
                'feature_dim': feature_dim,
            }, best_model_path)
        else:
            bad_epochs += 1

        print(f"Fold {fold_id} | Epoch {epoch+1}/{num_epochs} | Train {train_loss:.6f} | Val {val_loss:.6f} | Best {best_val:.6f}")

        if bad_epochs >= patience:
            print(f"Fold {fold_id} early stopping triggered at epoch {epoch+1} (best epoch {best_epoch}).")
            break

    return best_val, fold_history

In [ ]:
if __name__ == "__main__":
    main()

Fold 1: train_time_ids=5..5239  val_time_ids=5266..10789  train_n=71454 val_n=71446


Fold 1 | Epoch 1/6 | Train 0.000961 | Val 0.000008
Fold 1 | Epoch 2/6 | Train 0.000063 | Val 0.000003
Fold 1 | Epoch 3/6 | Train 0.000021 | Val 0.000003
Fold 1 | Epoch 4/6 | Train 0.000013 | Val 0.000002
Fold 1 | Epoch 5/6 | Train 0.000010 | Val 0.000002
Fold 1 | Epoch 6/6 | Train 0.000010 | Val 0.000002
Fold 2: train_time_ids=5..10779  val_time_ids=10793..15853  train_n=142900 val_n=71453


Fold 2 | Epoch 1/6 | Train 0.000717 | Val 0.000003
Fold 2 | Epoch 2/6 | Train 0.000029 | Val 0.000002
Fold 2 | Epoch 3/6 | Train 0.000015 | Val 0.000002
Fold 2 | Epoch 4/6 | Train 0.000008 | Val 0.000003
Fold 2 | Epoch 5/6 | Train 0.000007 | Val 0.000002
Fold 2 | Epoch 6/6 | Train 0.000006 | Val 0.000002
Fold 3: train_time_ids=5..15846  val_time_ids=15858..21148  train_n=214353 val_n=71452


Fold 3 | Epoch 1/6 | Train 0.000338 | Val 0.000004
Fold 3 | Epoch 2/6 | Train 0.000023 | Val 0.000002
Fold 3 | Epoch 3/6 | Train 0.000010 | Val 0.000002
Fold 3 | Epoch 4/6 | Train 0.000005 | Val 0.000002
Fold 3 | Epoch 5/6 | Train 0.000004 | Val 0.000002
Fold 3 | Epoch 6/6 | Train 0.000004 | Val 0.000002
Fold 4: train_time_ids=5..21139  val_time_ids=21164..26830  train_n=285805 val_n=71451


Fold 4 | Epoch 1/6 | Train 0.000414 | Val 0.000045


In [ ]:
# Smoke-test settings reference (kept separate on purpose).
SMOKE_HYPERPARAMETERS = {
    'batch_size': 64,
    'num_workers': 2,
}

SMOKE_CV_CONFIG = {
    'n_splits': 1,
    'gap_time_ids': 1,
    'num_epochs': 1,
    'stats_max_samples': 512,
}

def apply_smoke_settings(hparams, cfg):
    hparams.update(SMOKE_HYPERPARAMETERS)
    cfg.update(SMOKE_CV_CONFIG)
    print('Applied smoke settings:', hparams, cfg)

# To use later, run: apply_smoke_settings(hyperparameters, cv_config)

In [12]:
# Fold variance diagnostic: use this after cross-validation has run.
# It helps decide whether to increase the number of time-series splits.
import os
import numpy as np
import pandas as pd

scores_path = f"{DIR}/cv_scores.csv"

if 'cv_df' in globals() and isinstance(cv_df, pd.DataFrame) and not cv_df.empty:
    df_scores = cv_df.copy()
elif os.path.exists(scores_path):
    df_scores = pd.read_csv(scores_path)
else:
    raise FileNotFoundError(
        f"No fold scores found. Run training first or ensure {scores_path} exists."
    )

if 'best_val_loss' not in df_scores.columns:
    raise ValueError("Expected a 'best_val_loss' column in fold score data.")

losses = df_scores['best_val_loss'].astype(float).to_numpy()
mean_loss = float(np.mean(losses))
std_loss = float(np.std(losses, ddof=1)) if losses.size > 1 else 0.0
cv_ratio = float(std_loss / (mean_loss + 1e-12))
spread_ratio = float((np.max(losses) - np.min(losses)) / (mean_loss + 1e-12))

print('Fold Stability Diagnostic')
print(f"n_folds={len(losses)}")
print(f"mean_best_val_loss={mean_loss:.8f}")
print(f"std_best_val_loss={std_loss:.8f}")
print(f"cv_ratio={cv_ratio:.4f}")
print(f"min_max_spread_ratio={spread_ratio:.4f}")

if len(losses) < 5:
    recommendation = 'Increase to at least 5-6 folds for a more reliable estimate.'
elif cv_ratio <= 0.10 and spread_ratio <= 0.25:
    recommendation = 'Current folds look stable. Keeping 5-6 folds is reasonable.'
elif cv_ratio <= 0.20 and spread_ratio <= 0.40:
    recommendation = 'Moderate variability. Consider 8 folds if compute budget allows.'
else:
    recommendation = 'High variability. Consider 8-10 folds or revisiting split design/features.'

print(f"Recommendation: {recommendation}")

Fold Stability Diagnostic
n_folds=1
mean_best_val_loss=0.00000157
std_best_val_loss=0.00000000
cv_ratio=0.0000
min_max_spread_ratio=0.0000
Recommendation: Increase to at least 5-6 folds for a more reliable estimate.


In [ ]:
# Export trained checkpoints to your local machine after training finishes.
# This zips all fold-best checkpoints and downloads the archive from Colab.
import os
import glob
import shutil
from google.colab import files

checkpoint_dir = DIR
checkpoint_pattern = os.path.join(checkpoint_dir, 'best_model_fold_*.pth')
checkpoint_files = sorted(glob.glob(checkpoint_pattern))

if not checkpoint_files:
    raise FileNotFoundError(f'No fold checkpoints found at {checkpoint_pattern}. Run training first.')

export_dir = '/content/model_export'
os.makedirs(export_dir, exist_ok=True)

archive_base = os.path.join(export_dir, 'trained_model_checkpoints')
archive_path = shutil.make_archive(archive_base, 'zip', root_dir=checkpoint_dir, base_dir='.', verbose=False, logger=None)

print('Found checkpoints:')
for path in checkpoint_files:
    print(path)

print(f'Created archive: {archive_path}')
files.download(archive_path)

# Optional: copy to Drive as well if you want a persistent cloud copy.
# drive_copy_dir = '/content/drive/MyDrive/TimeSeriesDeepLearning_FIM601/model_exports'
# os.makedirs(drive_copy_dir, exist_ok=True)
# shutil.copy2(archive_path, os.path.join(drive_copy_dir, os.path.basename(archive_path)))